# Generating type 4 clones with LLMs

In [1]:
import os, re, json, textwrap, tempfile, subprocess, sys, uuid, random
import requests

DATASET_PATH = "../dataset/bigcodebench_normalized.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json"
OLLAMA_MODEL = "llama3.1:latest"   

# Generation settings
LLM_OPTS = {
    "temperature": 0.6,
    "top_p": 0.95,
    "repeat_penalty": 1.05,
    "num_predict": 768,   
}

N_ENTRIES = 4
CLONES_PER_ENTRY = 2  

In [2]:
def call_ollama_chat(messages, model=OLLAMA_MODEL, options=LLM_OPTS):
    """
    Call Ollama's /api/chat with role-based messages.
    Returns raw string content from the assistant.
    """
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": messages,
            "stream": False,
            "options": options
        },
        timeout=600
    )
    resp.raise_for_status()
    data = resp.json()
    return data["message"]["content"]

def extract_python_code(text: str) -> str:
    """
    Extract the first ```python ... ``` fenced block;
    if none found, return the whole text.
    """
    m = re.search(r"```python\s*(.*?)```", text, flags=re.S)
    if m:
        return m.group(1).strip()
    m = re.search(r"```\s*(.*?)```", text, flags=re.S) 
    return (m.group(1).strip() if m else text.strip())

def force_function_name(code: str, expected="task_func"):
    """
    Ensure the function is named `expected`.
    If the model wrote a different name, rename the top-level function.
    """
    import ast, astor
    try:
        tree = ast.parse(textwrap.dedent(code))
        for node in tree.body:
            if isinstance(node, ast.FunctionDef):
                node.name = expected
                break
        ast.fix_missing_locations(tree)
        return astor.to_source(tree)
    except Exception:
        return code  # if parsing fails, return as-is; validation will catch issues

def validate_with_unittest(code: str, tests: list) -> bool:
    """
    Write `code` + `tests` into a temp .py and execute it as a script.
    Returns True if all tests pass.
    """
    code_d = textwrap.dedent(code)
    tests_d = "\n\n".join(textwrap.dedent(t) for t in tests)

    full_script = f"""
{code_d}

{tests_d}

if __name__ == "__main__":
    import unittest
    unittest.main()
"""
    try:
        with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
            f.write(full_script)
            tmp = f.name

        res = subprocess.run(
            [sys.executable, tmp],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        os.remove(tmp)

        if res.returncode == 0:
            return True
        else:
            print("⚠️ Unit tests failed:\n", res.stdout, "\n--- STDERR ---\n", res.stderr)
            return False
    except Exception as e:
        print("Validation error:", e)
        return False


In [3]:
STRATEGY_HINTS = [
    # Keep behavior identical; do shallow refactors
    "- rename locals & args; reorder independent statements; introduce small helper vars; keep library calls and side-effects identical.",
    "- replace simple loops with list/dict comprehensions where safe; adjust arithmetic with equivalent identities; keep signature & imports.",
    "- wrap small expressions into temporary variables; change exception handling style without changing raised exceptions.",
]

SYSTEM_PROMPT = """You are a careful Python refactoring engine.
You produce a semantically equivalent variant (Type-4 clone) of the given function.
Rules:
- Output ONLY Python code in a single fenced block.
- Define exactly one function named `task_func` with the correct signature for the tests.
- Keep the same external behavior, side-effects, and library usage (imports allowed).
- Do NOT hardcode any test data or specific URLs or values from tests.
- Keep I/O contract identical (same return types, shapes, and exceptions).
"""

def build_user_prompt(original_body: str, description: str, libs: list, tests_snippet: str, strategy_hint: str) -> str:
    return f"""
You will be shown:
1) A short description and allowed libraries.
2) The original function BODY (not including the def line).
3) An excerpt of the unit tests (for signature and behavior cues). Do not overfit.

Description:
{description}

Allowed/expected libraries (may import as needed): {libs}

Original function BODY (indentation represents inside the function):
{textwrap.dedent(original_body).strip()}

Unit test excerpt (do not hardcode values; just infer signature/contract):
{textwrap.shorten(textwrap.dedent(tests_snippet), width=2000, placeholder=" ... ")}


Your task:
- Emit a semantically equivalent implementation named `task_func`.
- Keep side effects and external calls intact where visible (e.g., urllib/os/json/pandas usage).
- {strategy_hint}

Return ONLY the code in a single ```python fenced block.
"""


## Running the tests on the clones

In [4]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

sample = data[:N_ENTRIES]

results = []
for i, entry in enumerate(sample, 1):
    print(f"\n=== Entry {i}/{len(sample)} | id={entry['id']} ===")
    clones = []

    original_body = entry["original_code"]
    tests_list    = entry["test"]
    description   = entry.get("description", "")
    libs          = entry.get("metadata", {}).get("libs", [])

    # For the prompt, we can include the first test block (usually contains the signature/patches)
    tests_snippet = tests_list[0] if tests_list else ""

    for k in range(CLONES_PER_ENTRY):
        hint = STRATEGY_HINTS[k % len(STRATEGY_HINTS)]
        user_prompt = build_user_prompt(original_body, description, libs, tests_snippet, hint)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ]

        try:
            raw = call_ollama_chat(messages)
            code = extract_python_code(raw)

            # Ensure function is named task_func
            code = force_function_name(code, expected="task_func")

            ok = validate_with_unittest(code, tests_list)
            status = "✅ PASS" if ok else "❌ FAIL"
            print(f"  Clone {k+1}: {status}")

            if ok:
                clones.append({
                    "transformation": f"LLM/{OLLAMA_MODEL}",
                    "strategy_hint": hint,
                    "code": code
                })
        except Exception as e:
            print(f"  Error generating clone {k+1}: {e}")

    results.append({
        "id": entry["id"],
        "language": entry["language"],
        "description": description,
        "metadata": entry.get("metadata", {}),
        "original_code": original_body,
        "test": tests_list,
        "clones": clones
    })

# Save just the processed subset
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\nDone. Saved {len(results)} entries to {OUT_PATH}")



=== Entry 1/4 | id=b02e731e-cd87-4d49-9bf5-8a4e1246f246 ===
⚠️ Unit tests failed:
  
--- STDERR ---
 ....F....F
FAIL: test_large_list_with_seed (__main__.TestCases.test_large_list_with_seed)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\lamp6\AppData\Local\Temp\tmp2mu0x0ig.py", line 67, in test_large_list_with_seed
    self.assertAlmostEqual(result, 33.0, delta=0.5)  # This expected value should be calculated beforehand
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 1.0471781305114639e-05 != 33.0 within 0.5 delta (32.9999895282187 difference)

FAIL: test_specific_value_with_seed (__main__.TestCases.test_specific_value_with_seed)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\lamp6\AppData\Local\Temp\tmp2mu0x0ig.py", line 62, in test_specific_value_with_seed
    self.assertAlmostEqual(result, 2.5, delta=0.5)  # Thi